In [2]:
import csv
import json
import re
from difflib import SequenceMatcher

def parse_ground_truth_movies(ground_truth):
    """
    Parse ground truth that may contain multiple movies separated by commas.
    Extract clean movie titles from entries like "Movie1 (year), Movie2 (year)"
    """
    movies = []
    # Split by comma and clean each title
    movie_parts = ground_truth.split(',')
    
    for part in movie_parts:
        part = part.strip()
        # Remove year in parentheses like "(1984)"
        clean_title = re.sub(r'\s*\(\d{4}\)\s*', '', part).strip()
        if clean_title:
            movies.append(clean_title)
    
    return movies

def normalize_title(title):
    """
    Normalize movie title by removing common articles and punctuation.
    """
    # Remove articles and common words
    title = re.sub(r'\b(the|a|an)\b', '', title, flags=re.IGNORECASE)
    # Remove punctuation and extra spaces
    title = re.sub(r'[^\w\s]', '', title)
    # Remove extra whitespace
    title = re.sub(r'\s+', ' ', title).strip()
    return title.lower()

def similarity_score(a, b):
    """Calculate similarity between two strings."""
    return SequenceMatcher(None, a, b).ratio()

def extract_potential_titles(text):
    """
    Extract potential movie titles from text (quoted strings, capitalized phrases).
    """
    titles = []
    
    # Find quoted strings
    quoted_matches = re.findall(r'"([^"]*)"', text)
    titles.extend(quoted_matches)
    
    # Find phrases with QUOTATION_MARK format
    quotation_matches = re.findall(r'QUOTATION_MARK([^Q]+)QUOTATION_MARK', text)
    titles.extend(quotation_matches)
    
    # Find capitalized phrases (potential titles)
    capitalized_matches = re.findall(r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b', text)
    titles.extend(capitalized_matches)
    
    return [title.strip() for title in titles if len(title.strip()) > 2]

def count_questions_in_turn(content):
    """
    Count the number of questions in a single assistant turn.
    Returns 1 if any question is found, 0 otherwise (max 1 per turn).
    """
    # Look for question marks
    question_marks = content.count('?')
    
    # Look for question patterns (What, How, When, Where, Why, etc.)
    question_patterns = [
        r'\b(what|how|when|where|why|who|which|can|could|would|will|do|does|did|have|has|had|is|are|was|were)\b[^.!?]*\?',
        r'\b(any suggestions|recommendations|thoughts|ideas)\b',
        r'\blet me know\b'
    ]
    
    pattern_matches = 0
    for pattern in question_patterns:
        if re.search(pattern, content, re.IGNORECASE):
            pattern_matches += 1
            break  # Only count one question per turn
    
    # Return 1 if any questions found, 0 otherwise
    return 1 if (question_marks > 0 or pattern_matches > 0) else 0

def check_movie_in_turn(ground_truth_movies, turn_content, similarity_threshold=0.7):
    """
    Check if any ground truth movie is mentioned in a specific turn.
    """
    turn_lower = turn_content.lower()
    normalized_turn = normalize_title(turn_content)
    potential_titles = extract_potential_titles(turn_content)
    
    for movie in ground_truth_movies:
        movie_lower = movie.lower()
        normalized_movie = normalize_title(movie)
        
        # Direct substring match
        if movie_lower in turn_lower:
            return True, movie
        
        # # Normalized title match
        # if normalized_movie in normalized_turn:
        #     return True, movie
        
        # # Similarity match with potential titles
        # for potential_title in potential_titles:
        #     if similarity_score(movie_lower, potential_title.lower()) >= similarity_threshold:
        #         return True, movie
        #     if similarity_score(normalized_movie, normalize_title(potential_title)) >= similarity_threshold:
        #         return True, movie
        
        # # Partial word match
        # movie_words = set(normalized_movie.split())
        # if len(movie_words) > 1:
        #     turn_words = set(normalized_turn.split())
        #     word_overlap = len(movie_words.intersection(turn_words))
        #     if word_overlap >= len(movie_words) * 0.75:
        #         return True, movie
    
    return False, None

def analyze_conversation_metrics(csv_file_path, similarity_threshold=0.7):
    """
    Analyze conversation metrics:
    1. Count questions asked by assistant (max 1 per turn)
    2. Count turns to recommend ground truth
    """
    results = []
    
    try:
        with open(csv_file_path, 'r', encoding='utf-8') as file:
            reader = csv.DictReader(file)
            
            for row in reader:
                dialog_id = row['dialog_id']
                ground_truth = row['ground_truth'].strip()
                generated_conversation = row['generated_conversation']
                
                # Parse ground truth movies
                ground_truth_movies = parse_ground_truth_movies(ground_truth)
                
                try:
                    conversation = json.loads(generated_conversation)
                    
                    # Initialize metrics
                    total_questions = 0
                    turns_to_recommendation = None
                    assistant_turn_count = 0
                    
                    # Analyze each message
                    for message in conversation:
                        if message.get('role') == 'assistant':
                            assistant_turn_count += 1
                            content = message.get('content', '')
                            
                            # Count questions (max 1 per turn)
                            questions_in_turn = count_questions_in_turn(content)
                            total_questions += questions_in_turn
                            
                            # Check if ground truth movie mentioned
                            if turns_to_recommendation is None:
                                movie_mentioned, mentioned_movie = check_movie_in_turn(
                                    ground_truth_movies, content, similarity_threshold
                                )
                                if movie_mentioned:
                                    turns_to_recommendation = assistant_turn_count
                    
                    # If no ground truth movie was ever mentioned
                    if turns_to_recommendation is None:
                        turns_to_recommendation = -1  # Indicates never mentioned
                    
                    results.append({
                        'dialog_id': dialog_id,
                        'ground_truth': ground_truth,
                        'ground_truth_movies': ground_truth_movies,
                        'total_questions': total_questions,
                        'turns_to_recommendation': turns_to_recommendation,
                        'total_assistant_turns': assistant_turn_count,
                        'ground_truth_mentioned': turns_to_recommendation > 0
                    })
                    
                except json.JSONDecodeError:
                    print(f"Error parsing JSON for dialog_id: {dialog_id}")
                    results.append({
                        'dialog_id': dialog_id,
                        'ground_truth': ground_truth,
                        'error': 'JSON parsing error'
                    })
    
    except FileNotFoundError:
        print(f"Error: File '{csv_file_path}' not found.")
        return None
    except Exception as e:
        print(f"Error reading file: {str(e)}")
        return None
    
    return results

def print_conversation_metrics(results):
    """Print conversation metrics analysis."""
    if results is None:
        return
    
    # Filter out error entries
    valid_results = [r for r in results if 'error' not in r]
    error_count = len(results) - len(valid_results)
    
    if not valid_results:
        print("No valid entries to analyze.")
        return
    
    # Calculate statistics
    total_entries = len(valid_results)
    
    # Question statistics
    total_questions = sum(r['total_questions'] for r in valid_results)
    avg_questions = total_questions / total_entries
    max_questions = max(r['total_questions'] for r in valid_results)
    min_questions = min(r['total_questions'] for r in valid_results)
    
    # Recommendation turn statistics
    successful_recommendations = [r for r in valid_results if r['turns_to_recommendation'] > 0]
    failed_recommendations = [r for r in valid_results if r['turns_to_recommendation'] == -1]
    
    success_rate = (len(successful_recommendations) / total_entries * 100) if total_entries > 0 else 0
    
    if successful_recommendations:
        avg_turns_to_rec = sum(r['turns_to_recommendation'] for r in successful_recommendations) / len(successful_recommendations)
        max_turns_to_rec = max(r['turns_to_recommendation'] for r in successful_recommendations)
        min_turns_to_rec = min(r['turns_to_recommendation'] for r in successful_recommendations)
    else:
        avg_turns_to_rec = max_turns_to_rec = min_turns_to_rec = 0
    
    print("=" * 70)
    print("CONVERSATION METRICS ANALYSIS")
    print("=" * 70)
    print(f"Total entries analyzed: {total_entries}")
    if error_count > 0:
        print(f"Entries with errors: {error_count}")
    print()
    
    print("QUESTION METRICS:")
    print(f"  Total questions asked by assistant: {total_questions}")
    print(f"  Average questions per conversation: {avg_questions:.2f}")
    print(f"  Max questions in one conversation: {max_questions}")
    print(f"  Min questions in one conversation: {min_questions}")
    print()
    
    print("RECOMMENDATION METRICS:")
    print(f"  Conversations with ground truth mentioned: {len(successful_recommendations)} ({success_rate:.1f}%)")
    print(f"  Conversations without ground truth: {len(failed_recommendations)} ({100-success_rate:.1f}%)")
    print()
    
    if successful_recommendations:
        print("TURNS TO RECOMMENDATION (for successful cases):")
        print(f"  Average turns to recommend: {avg_turns_to_rec:.2f}")
        print(f"  Fastest recommendation (turns): {min_turns_to_rec}")
        print(f"  Slowest recommendation (turns): {max_turns_to_rec}")
        print()
        
        # Distribution of turns to recommendation
        turn_distribution = {}
        for r in successful_recommendations:
            turns = r['turns_to_recommendation']
            turn_distribution[turns] = turn_distribution.get(turns, 0) + 1
        
        print("Distribution of turns to recommendation:")
        for turns in sorted(turn_distribution.keys()):
            count = turn_distribution[turns]
            percentage = (count / len(successful_recommendations) * 100)
            print(f"  Turn {turns}: {count} conversations ({percentage:.1f}%)")
    
    print()
    print("SAMPLE RESULTS (first 5):")
    for i, result in enumerate(valid_results[:5]):
        status = f"Recommended in turn {result['turns_to_recommendation']}" if result['turns_to_recommendation'] > 0 else "Never recommended"
        print(f"  {i+1}. {result['dialog_id']}")
        print(f"     Questions: {result['total_questions']}, {status}")
        print(f"     Ground truth: {result['ground_truth']}")

# Example usage
if __name__ == "__main__":
    # Replace with your actual CSV file path
    dataset = "redial"
    alg = "sft"
    model = 'llama3-2-1b-instruct'
    folder_path = f"../generated_testsets/multiturn_test/{alg}/{model}/{dataset}/"
    csv_file_path = folder_path + "generated_movie_conversations2.csv"  # Your input CSV file
    
    results = analyze_conversation_metrics(csv_file_path)
    print_conversation_metrics(results)

CONVERSATION METRICS ANALYSIS
Total entries analyzed: 1076

QUESTION METRICS:
  Total questions asked by assistant: 3591
  Average questions per conversation: 3.34
  Max questions in one conversation: 5
  Min questions in one conversation: 0

RECOMMENDATION METRICS:
  Conversations with ground truth mentioned: 272 (25.3%)
  Conversations without ground truth: 804 (74.7%)

TURNS TO RECOMMENDATION (for successful cases):
  Average turns to recommend: 2.75
  Fastest recommendation (turns): 1
  Slowest recommendation (turns): 5

Distribution of turns to recommendation:
  Turn 1: 59 conversations (21.7%)
  Turn 2: 61 conversations (22.4%)
  Turn 3: 71 conversations (26.1%)
  Turn 4: 50 conversations (18.4%)
  Turn 5: 31 conversations (11.4%)

SAMPLE RESULTS (first 5):
  1. 20001
     Questions: 1, Never recommended
     Ground truth: Police Academy  (1984), Police Academy 2: Their First Assignment (1985), Lethal Weapon (1987)
  2. 20047
     Questions: 2, Recommended in turn 1
     Ground t